In [1]:
import torch
import numpy as np

In [3]:
a = np.random.randn(1, 12)

In [6]:
a

array([[ 0.54107603,  0.31111095, -1.27236695, -1.19106203,  1.1616997 ,
        -0.38486562, -1.61442471,  0.37846794,  1.26343258, -0.4810045 ,
         0.34556866,  1.31376353]])

In [5]:
np.split(a, 3, axis=-1)

[array([[ 0.54107603,  0.31111095, -1.27236695, -1.19106203]]),
 array([[ 1.1616997 , -0.38486562, -1.61442471,  0.37846794]]),
 array([[ 1.26343258, -0.4810045 ,  0.34556866,  1.31376353]])]

In [41]:
torch.manual_seed(42)
a = torch.randn(1, 4, 1, 3).numpy()
d = torch.randn(3).numpy()
b = torch.randn(5, 4, 2, 3).numpy()

In [42]:
a

array([[[[ 0.33669037,  0.1288094 ,  0.23446237]],

        [[ 0.23033303, -1.1228564 , -0.18632829]],

        [[ 2.2082014 , -0.63799703,  0.46165723]],

        [[ 0.26735088,  0.53490466,  0.8093572 ]]]], dtype=float32)

In [43]:
c = d + b 

In [46]:
db, bb = np.broadcast_arrays(d, b)
ab, bb = np.broadcast_arrays(a, b)

In [50]:
broadcast = np.broadcast(a, b)

In [47]:
db.shape

(5, 4, 2, 3)

In [48]:
ab.shape

(5, 4, 2, 3)

In [39]:
c_grad = np.ones_like(c)

In [63]:
a.shape

(1, 4, 1, 3)

In [64]:
b.shape

(5, 4, 2, 3)

In [83]:
from typing import Any

def _sum_to_shape(grad: np.ndarray[Any, Any], target_shape: tuple[int, ...]) -> np.ndarray[Any, Any]:
    """Sum gradient over broadcasted dimensions to match target shape."""
    if grad.shape == target_shape:
        return grad

    # Handle leading dimensions that don't exist in target: e.g. shapes [2] to [3, 2]
    # Afterwards, grad has the same number of dimensions as the target_shape -> len(grad.shape) == len(target_shape)
    for _ in range(len(grad.shape) - len(target_shape)):
        grad = np.sum(grad, axis=0)

    # Handle dimensions where target has size 1 but grad doesn't
    sum_axes: list[int] = []
    for axis, (target_size, grad_size) in enumerate(zip(target_shape, grad.shape)):
        if target_size == 1 and grad_size != 1:
            sum_axes.append(axis)

    if sum_axes:
        grad = np.sum(grad, axis=tuple(sum_axes), keepdims=True)

    return grad

In [76]:
grad = c_grad

In [78]:
for _ in range(len(grad.shape) - len([1, 4, 1, 3])):
    grad = np.sum(grad, axis=0)

In [67]:
c_grad.shape

(5, 4, 2, 3)

In [93]:
_sum_to_shape(c_grad, [1, 4, 1, 3]).shape

(1, 4, 1, 3)

In [82]:
grad.sum(axis=(0, 2), keepdims=True).shape

(1, 4, 1, 3)

In [2]:
import numpy as np
from typing import Any, Sequence
from types import EllipsisType

In [3]:
def split_numpy(
    arr: np.ndarray[Any, Any],
    split_size_or_sections: int | Sequence[int | slice | EllipsisType],
    axis: int = 0,
    append_remainder: bool = False
) -> list[np.ndarray[Any, Any]]:
    """Split the array into a list of arrays.

    Usage:
    >>> arr = np.array([1, 2, 3, 4])
    >>> split_numpy(arr, 2)
    [array([1, 2]), array([3, 4])]

    
    >>> arr = np.array([1, 2, 3, 4, 5])
    >>> split_numpy(arr, [2, 3])
    [array([1, 2]), array([3, 4, 5])]

    >>> arr = np.array([1, 2, 3, 4, 5])
    >>> split_numpy(arr, [..., 2])
    [array([1, 2, 3]), array([4, 5])]
    """
    if isinstance(split_size_or_sections, int):
        s = arr.shape[axis] / split_size_or_sections
        if s.is_integer():
            s = int(s)
        else:
            raise ValueError(f'arr shape[{axis}] ({arr.shape[axis]}) is not divisable by {split_size_or_sections}')
        return np.split(arr, s, axis=axis)
    else:
        r = [s == Ellipsis for s in split_size_or_sections]
        c = np.count_nonzero(r)
        if c > 1:
            raise ValueError('Ellipsis ... is used more than once in sections')
        elif c == 1:
            s = np.array(split_size_or_sections)
            s[r] = 0
            s[r] = arr.shape[axis] - s.sum()
        else:
            s = split_size_or_sections

        s = np.cumsum(s)  # type: ignore
        splits = np.split(arr, s, axis=axis)  # type: ignore
        del splits[-1]
        return splits

In [11]:
arr = np.array([1, 2, 3, 4, 5])
split_numpy(arr, [..., 2])

[array([1, 2, 3]), array([4, 5])]

In [13]:
0x8ff3

36851

In [14]:
2**16

65536

In [6]:
import numpy as np

def var(x: np.ndarray, dim: int | None = None, keepdims: bool = False) -> np.ndarray:
    mean = x.mean(axis=dim, keepdims=keepdims)
    N = x.shape[dim]
    
    return ((x - mean)**2).sum(axis=dim, keepdims=keepdims) / N

In [12]:
import torch

torch.manual_seed(42)
a = torch.randn(3, 4).numpy()
a

array([[ 0.33669037,  0.1288094 ,  0.23446237,  0.23033303],
       [-1.1228564 , -0.18632829,  2.2082014 , -0.63799703],
       [ 0.46165723,  0.26735088,  0.53490466,  0.8093572 ]],
      dtype=float32)

In [13]:
var(a, -1, keepdims=True)

array([[0.00540397],
       [1.6404214 ],
       [0.03779347]], dtype=float32)

In [14]:
a.var(axis=-1, keepdims=True)

array([[0.00540397],
       [1.6404214 ],
       [0.03779347]], dtype=float32)